# Fill Template: Fossil Layers Visualization

This notebook overlays the fossil layers scatter tree visualization onto the WSE layer phylo template PDF.

The approach:
1. Read the positions PDF to find the `#deadbe`-colored rectangle
2. Read the scatter tree visualization PDF and rotate it 180 degrees
3. Overlay the rotated visualization onto the template PDF at the rectangle's position

In [ ]:
import glob
import os

import fitz  # PyMuPDF

## Configure Paths

In [ ]:
positions_pdf_path = "assets/wse-layer-phylo-positions.pdf"
template_pdf_path = "assets/wse-layer-phylo-template.pdf"
scatter_tree_path = glob.glob(
    "teeplots/2026-03-04-fossil-layers/"
    "hue=layer+layout=vertical+regime=pure+viz=draw-scatter-tree+ext=.pdf",
)[0]
output_path = "outplots/wse-layer-phylo-fossil-layers.pdf"

## Find `#deadbe` Rectangle Position

In [ ]:
def find_deadbe_rect(pdf_path: str) -> fitz.Rect:
    """Find the rectangle filled with color #deadbe in the positions PDF."""
    doc = fitz.open(pdf_path)
    page = doc[0]

    # Target color #deadbe normalized to [0, 1]
    target_r = 0xDE / 255.0
    target_g = 0xAD / 255.0
    target_b = 0xBE / 255.0

    found_rect = None

    # Extract drawing commands to find rectangles with the target fill color
    paths = page.get_drawings()
    for path in paths:
        fill = path.get("fill")
        if fill is None:
            continue
        r, g, b = fill[0], fill[1], fill[2]
        # Check if fill color matches #deadbe (with tolerance)
        if (
            abs(r - target_r) < 0.02
            and abs(g - target_g) < 0.02
            and abs(b - target_b) < 0.02
        ):
            found_rect = fitz.Rect(path["rect"])
            break

    doc.close()

    if found_rect is None:
        raise ValueError(f"No #deadbe rectangle found in {pdf_path}")

    return found_rect


target_rect = find_deadbe_rect(positions_pdf_path)
print(f"Found #deadbe rectangle at: {target_rect}")
print(f"Width: {target_rect.width:.1f}, Height: {target_rect.height:.1f}")

## Fill Template with Rotated Scatter Tree

In [ ]:
# Open the template and scatter tree PDFs
template_doc = fitz.open(template_pdf_path)
scatter_doc = fitz.open(scatter_tree_path)

template_page = template_doc[0]
scatter_page = scatter_doc[0]

# Create a pixmap of the scatter tree page, rotated 180 degrees
# rotate=180 flips the page upside down
mat = fitz.Matrix(1, 1).prerotate(180)

# Render scatter tree to a pixmap at high resolution
pix = scatter_page.get_pixmap(matrix=fitz.Matrix(4, 4).prerotate(180), alpha=False)

# Insert the rotated scatter tree image into the template at the target rectangle
template_page.insert_image(target_rect, pixmap=pix)

# Save the result
os.makedirs(os.path.dirname(output_path), exist_ok=True)
template_doc.save(output_path)

template_doc.close()
scatter_doc.close()

print(f"Saved filled template to: {output_path}")

## Verify Output

In [ ]:
# Quick verification: open the output and check it has content
verify_doc = fitz.open(output_path)
verify_page = verify_doc[0]
print(f"Output page size: {verify_page.rect.width:.0f} x {verify_page.rect.height:.0f}")

# Display as inline image for visual verification
from IPython.display import display, Image

pix = verify_page.get_pixmap(matrix=fitz.Matrix(2, 2))
display(Image(data=pix.tobytes("png")))

verify_doc.close()